# `llm_query_router` (LlmQueryRouterModule)
- **Category**: Logic (LLM)
- **Role**: 질문 의도를 파악하여 연관 기업명과 대상 재무제표 시트명(손익계산서, 재무상태표, 현금흐름표 등)을 사전 라우팅합니다.


In [ ]:
import sys
from pathlib import Path
import json

# 프로젝트 루트 경로 등록
PROJECT_ROOT = Path(".").resolve().parent.parent if Path(".").resolve().name == "modules" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def print_io(title: str, input_data: dict, output_data: dict):
    print("=" * 70)
    print(f"📌 [Module Execution] {title}")
    print("=" * 70)
    print("\n📥 [Input DTO]")
    print(json.dumps(input_data, indent=2, ensure_ascii=False))
    print("\n📤 [Output Result]")
    print(json.dumps(output_data, indent=2, ensure_ascii=False))
    print("\n")


In [ ]:
from unittest.mock import MagicMock
from modules.query.llm_query_router import LlmQueryRouterModule, LlmQueryRouterInputDTO, LlmQueryRouterConfigDTO
from backend.providers.llm.chat_completion import ChatCompletionResult

mock_router_llm = MagicMock()
mock_router_llm.complete_with_metadata.return_value = ChatCompletionResult(
    content=json.dumps({
        "matched": True,
        "company_name": "삼성전자",
        "sheets": ["손익계산서", "포괄손익계산서"],
        "confidence": 0.95,
        "reasoning": "영업이익 및 수익성 지표는 손익계산서 시트에 위치합니다."
    }),
    usage={"prompt_tokens": 30, "completion_tokens": 40, "total_tokens": 70},
    latency_seconds=0.22,
)

module = LlmQueryRouterModule(completion_client=mock_router_llm)

sample_input = {
    "query_context": {
        "question_id": "QUERY-001",
        "question_text": "삼성전자 손익계산서에서 영업이익 조회해줘"
    }
}
input_dto = LlmQueryRouterInputDTO(**sample_input)
output = module.run(input_dto, config=LlmQueryRouterConfigDTO(model="gpt-5.6-luna"))
print_io("llm_query_router (LlmQueryRouterModule)", sample_input, output)
